# Evaluation metrics vs code parameters (CPP experiments)

This notebook loads CSVs from `/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp` and plots **Y_METRICS** (relerr, spearman, recall, recon_error) vs:

- **Number of subquantizers** (grouped by nbits)
- **Bits per subvector (`nbits`)** (grouped by n_subquantizers)
- **Bits per vector**
- **Compression rate** (x=compression rate, y=chosen metric)

Set `Y_METRICS = ["relerr", "spearman", "recall", "recon_error"]` to plot all metrics. Each metric gets the same figure types (vs nbits, vs n_subq, compression_rate vs Y, etc.).

Figures are styled similarly to those in `DARTH_plus_conformal.ipynb` (font sizes, figure sizes, and general aesthetics).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, LogLocator, ScalarFormatter

# --- Plot style (aligned with DARTH_plus_conformal.ipynb) ---
# Base style - will be overridden to font.size=40 for individual plots
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 40,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 21,
})
# plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

DATA_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp")
CSV_PATTERN = "*_adc_vs_exact_eval.csv"

In [ ]:
from glob import glob

def load_relerr_data(data_dir: Path, pattern: str = CSV_PATTERN) -> pd.DataFrame:
    """Load and concatenate all relative-error CSVs in `data_dir`.

    Assumes filenames of the form `{dataset}_{method}_adc_vs_exact_eval.csv`.
    """
    csv_paths = sorted(data_dir.glob(pattern))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs matching {pattern!r} found in {data_dir}")

    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path)

        # Infer dataset/method from filename if needed
        name = path.stem  # e.g., deep_OPQ_adc_vs_exact_eval
        parts = name.split("_")
        if len(parts) >= 2:
            file_dataset, file_method = parts[0], parts[1]
            if "dataset" not in df.columns:
                df["dataset"] = file_dataset
            if "method" not in df.columns:
                df["method"] = file_method
        dfs.append(df)

    all_df = pd.concat(dfs, ignore_index=True)

    # Ensure expected columns exist
    required_cols = [
        "method",
        "dataset",
        "n_subquantizers",
        "nbits",
        "bits_per_vector",
        "rel_error_mean",
        "rel_error_std",
    ]
    missing = [c for c in required_cols if c not in all_df.columns]
    if missing:
        raise ValueError(f"Missing columns in concatenated DF: {missing}")

    return all_df


def _load_eval_csvs(data_dir: Path, pattern: str, suffix: str):
    """Load and concatenate CSVs matching pattern. Returns None if no files found."""
    csv_paths = sorted(data_dir.glob(pattern))
    if not csv_paths:
        return None
    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path)
        name = path.stem.replace(suffix, "")
        parts = name.split("_")
        if len(parts) >= 2:
            if "dataset" not in df.columns:
                df["dataset"] = parts[0]
            if "method" not in df.columns:
                df["method"] = parts[1]
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def build_plot_df(relerr_df: pd.DataFrame, data_dir: Path) -> pd.DataFrame:
    """Build unified dataframe with all metrics: relerr, compression_rate, spearman, recall, recon_error.

    Starts with relerr_df (has rel_error_mean) and merges in eval CSVs.
    Merges on (dataset, method, experiment_folder, n_subquantizers, nbits).
    """
    plot_df = relerr_df.copy()
    merge_cols = ["dataset", "method", "n_subquantizers", "nbits"]
    if "experiment_folder" in plot_df.columns:
        merge_cols.append("experiment_folder")

    for suffix, metric_cols in [
        ("_compression_rate", ["compression_rate"]),
        ("_reconstruction_error", ["reconstruction_error"]),
        ("_spearman", ["spearman"]),
        ("_recall", ["recall_1", "recall_10", "recall_100"]),
    ]:
        extra = _load_eval_csvs(data_dir, f"*{suffix}.csv", suffix)
        if extra is not None:
            add_cols = [c for c in metric_cols if c in extra.columns]
            if add_cols:
                cols = [c for c in merge_cols if c in extra.columns] + add_cols
                plot_df = plot_df.merge(extra[cols].drop_duplicates(), on=merge_cols, how="left")

    return plot_df


relerr_df = load_relerr_data(DATA_DIR)
relerr_df.head()

# ============================================================================
# CONFIGURATION: Select methods, datasets, and x-axis columns to plot
# ============================================================================

# Select which methods to plot (available: OPQ, PQ)
METHODS_TO_PLOT = ["PQ"]  # Change to ["OPQ"] or ["PQ"] to plot only one

# Select which datasets to plot (available: deep, gist)
DATASETS_TO_PLOT = ["deep", "bigann", "gist", "msmarco", "openai"]  # Change to ["deep"] or ["gist"] to plot only one
# DATASETS_TO_PLOT = ["deep"]  # Change to ["deep"] or ["gist"] to plot only one

# ADC CPU time unit: "ms" for milliseconds, "s" for seconds
ADC_TIME_UNIT = "s"

_adc_col = "adc_cpu_time_pp_ms" if ADC_TIME_UNIT == "ms" else "adc_cpu_time_pp"
_adc_label = f"ADC time ({ADC_TIME_UNIT})"

# Select which x-axis columns to plot (column_name, display_label)
# Available columns: n_subquantizers, nbits, bits_per_vector, train_time_s, or any other numeric column
X_COLUMNS_TO_PLOT = [
    # ("n_subquantizers", "Num subq"),
    # ("nbits", "nbits"),
    # ("bits_per_vector", "Bits per vector"),
    # ("train_time_s", "Training time (seconds)"),  # Avg Relative Error vs training time
    # ("adc_time_s", "ADC time (seconds)"),  # Avg Relative Error vs asymmetric distance computation time
    # ("distance_table_time_s", "Distance table time (seconds)"),  # Avg Relative Error vs distance table computation time
    # ("adc_time_2", "ADC time (s)"),  # distance_table_time_s + adc_time_s (wall clock)
    # ("adc_cpu_time", "ADC CPU time (s)"),  # total ADC CPU time (process_time)
    # ("adc_cpu_time_pp", "ADC CPU time per pair (s)"),  # per (query, db) pair
    (_adc_col, _adc_label),
]

# For nbits x-axis: which n_subquantizers (curves) to show per dataset.
# - dict: dataset name -> list of subq (e.g. {"deep": [1, 2, 4, 8, 16], "gist": [1, 4, 8]})
# - list: same list for all datasets (e.g. [1, 2, 4, 8, 16])
# - None: show all subq for all datasets
NBITS_PLOT_SUBQUANTIZERS = {"deep": [1, 4, 12, 24, 32, 96], "bigann": [1, 4, 16, 32, 64, 128], "gist": [1, 8, 40, 60, 320, 960], "msmarco": [1, 8, 32, 64, 256, 1024], "openai": [1, 32, 128, 256, 512, 1536]}  # or [1, 2, 4, 8] or None

# For Num subq x-axis: which n_subquantizers (x-axis points) to show per dataset.
# - dict: dataset name -> list of subq; list: same for all; None: show all
NUM_SUBQ_PLOT_SUBQUANTIZERS = {"deep": [1, 4, 12, 24, 32, 96], "bigann": [1, 4, 16, 32, 64, 128], "gist": [1, 8, 40, 60, 320, 960], "msmarco": [1, 8, 32, 64, 256, 1024], "openai": [1, 32, 128, 256, 512, 1536]}  # or [1, 2, 4, 8] or None

# You can also add custom columns, e.g.:
# X_COLUMNS_TO_PLOT = [
#     ("n_subquantizers", "Number of subquantizers"),
#     ("train_time_s", "Training time (seconds)"),
# ]

# Optional: Set y-axis limits for common x-axis columns
# Format: {column_name: (ymin, ymax)} or {column_name: None} to use auto
# This ensures all plots with the same x-axis have the same y-axis range for easy comparison
YLIM_CONFIG = {
    # "n_subquantizers": (0.0, 0.6),  # Example: set ylim for subquantizers plots
    # "nbits": (0.0, 0.3),             # Example: set ylim for nbits plots
    # "bits_per_vector": None,         # Use None or omit to use auto scaling
}

# Grouping configuration for bits_per_vector plots
# Options: "nbits" or "n_subquantizers"
# - "nbits": Groups by bits per subvector (shows how precision per subvector affects performance)
# - "n_subquantizers": Groups by number of subquantizers (shows trade-off between granularity and total bits)
BITS_PER_VECTOR_GROUPING = "n_subquantizers" # "n_subquantizers"  # Change to "nbits" to group by bits per subvector instead

# Additional plots with different y-axis columns
# Format: list of tuples (x_col, x_label, y_col, y_label, group_by)
# - x_col: Column name for x-axis
# - x_label: Display label for x-axis
# - y_col: Column name for y-axis (e.g., "adc_time_s", "rel_error_mean", "train_time_s")
# - y_label: Display label for y-axis (or None for auto-generated)
# - group_by: "nbits" or "n_subquantizers" - how to group the curves
ADDITIONAL_PLOTS = [
    # Plot ADC time vs number of subquantizers, grouped by nbits
    ("n_subquantizers", "Num subq", "adc_time_s", "ADC time (seconds)", "nbits"),
    # Add more plots here as needed
    # ("bits_per_vector", "Bits per vector", "adc_time_s", "ADC time (seconds)", "nbits"),
    
    # Training time plots
    # ("n_subquantizers", "Num subq", "train_time_s", "Training time (seconds)", "nbits"),
    # ("nbits", "nbits", "train_time_s", "Training time (seconds)", "n_subquantizers"),
    # ("bits_per_vector", "Bits per vector", "train_time_s", "Training time (seconds)", "nbits"),
]

# Bar plots for timing metrics (bin-style plots) with averaging
# Format: list of tuples (x_col, x_label, y_col, y_label, group_by)
# - x_col: Column name for x-axis (will be grouped into bins/bars)
# - x_label: Display label for x-axis
# - y_col: Column name for y-axis (e.g. "train_time_s", "adc_time_s", "distance_table_time_s")
# - y_label: Display label for y-axis (or None for auto-generated)
# - group_by: "nbits" or "n_subquantizers" - dimension to average over (the legend dimension)
BAR_PLOTS = [
    # Training time bar plots (averaged over legend dimension)
    # Plot 1: Training time vs Num subq (averaged over nbits) - single bars per subq
    # ("n_subquantizers", "Num subq", "train_time_s", "Training time (seconds)", "nbits"),
    # Plot 2: Training time vs nbits (averaged over n_subquantizers) - single bars per nbits
    # ("nbits", "nbits", "train_time_s", "Training time (seconds)", "n_subquantizers"),

    # ADC time bar plots (averaged over legend dimension)
    # Plot 3: ADC time vs Num subq (averaged over nbits)
    # ("n_subquantizers", "Num subq", "adc_time_s", "ADC time (seconds)", "nbits"),
    # Plot 4: ADC time vs nbits (averaged over n_subquantizers)
    # ("nbits", "nbits", "adc_time_s", "ADC time (seconds)", "n_subquantizers"),

    # Distance table time bar plots (averaged over legend dimension)
    # Plot 5: Distance table time vs Num subq (averaged over nbits)
    # ("n_subquantizers", "Num subq", "distance_table_time_s", "Distance table time (seconds)", "nbits"),
    # Plot 6: Distance table time vs nbits (averaged over n_subquantizers)
    # ("nbits", "nbits", "distance_table_time_s", "Distance table time (seconds)", "n_subquantizers"),
]

# Y-axis metrics to plot (applies to ALL plots: relerr vs nbits, vs n_subq, compression_rate vs Y, etc.)
# Options: relerr, spearman, recon_error, recall (or recall_1, recall_10, recall_100)
Y_METRICS = ["relerr"]  # e.g. ["relerr", "spearman", "recall", "recon_error"]

# Metric name -> (column name, display label)
Y_METRIC_MAP = {
    "relerr": ("rel_error_mean", "Avg Relative Error"),
    "recon_error": ("reconstruction_error", "Recon error"),
    "spearman": ("spearman", "Spearman cor"),
    "recall_1": ("recall_1", "Recall@1"),
    "recall_10": ("recall_10", "Recall@10"),
    "recall_100": ("recall_100", "Recall@100"),
}

# For compression_rate vs Y plots only: how to group curves
COMPRESSION_RATE_GROUPING = "nbits"  # or "n_subquantizers"

# Build unified plot_df with all metrics merged in
plot_df = build_plot_df(relerr_df, DATA_DIR)

# Prefer CPU-time columns when available (isolates from system load).
# Run: python scripts/evals/measure_adc_cpu_time.py  to populate these.
if "adc_time_cpu_s" in plot_df.columns:
    plot_df["adc_time_s"] = plot_df["adc_time_cpu_s"].combine_first(plot_df["adc_time_s"])
if "distance_table_time_cpu_s" in plot_df.columns:
    plot_df["distance_table_time_s"] = plot_df["distance_table_time_cpu_s"].combine_first(plot_df["distance_table_time_s"])
if "adc_cpu_time" in plot_df.columns:
    plot_df["adc_time_2"] = plot_df["adc_cpu_time"].combine_first(plot_df["adc_time_2"])

# Derive millisecond per-pair column (avoids scientific notation in plots)
if "adc_cpu_time_pp" in plot_df.columns:
    plot_df["adc_cpu_time_pp_ms"] = plot_df["adc_cpu_time_pp"] * 1e3

In [ ]:
# Color and marker palettes (will be dynamically assigned based on data)
# These are ordered lists that will be cycled through based on unique values in the data
COLOR_PALETTE = [
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red",
    "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan",
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red"
]

MARKER_PALETTE = [
    "o", "v", "s", "^", "D", "<", ">", "p", "*", "h", "H", "X", "d", "P", "8"
]

def create_dynamic_color_map(unique_values):
    """Create a color map dynamically based on unique values in the data."""
    sorted_values = sorted(unique_values)
    color_map = {}
    for i, val in enumerate(sorted_values):
        color_map[val] = COLOR_PALETTE[i % len(COLOR_PALETTE)]
    return color_map

def create_dynamic_marker_map(unique_values):
    """Create a marker map dynamically based on unique values in the data."""
    sorted_values = sorted(unique_values)
    marker_map = {}
    for i, val in enumerate(sorted_values):
        marker_map[val] = MARKER_PALETTE[i % len(MARKER_PALETTE)]
    return marker_map


def plot_relerr_vs_x(
    df: pd.DataFrame, 
    x_col: str, 
    x_label: str, 
    y_col: str = "rel_error_mean",
    y_label: str = None,
    methods: list = None,
    datasets: list = None,
    ylim: tuple = None,
    group_by: str = None,
    output_dir: Path = None,
    nbits_subquantizers=None,
    num_subq_plot_subquantizers=None,
):
    """
    Plot `y_col` vs `x_col`, with separate figures per method and dataset.

    Grouping logic:
    - When x_col is 'n_subquantizers': separate curves by nbits.
    - When x_col is 'nbits': separate curves by n_subquantizers.
    - For x_col 'nbits', nbits_subquantizers restricts which subq curves are shown: None = all;
      list = same subq for all datasets; dict = dataset -> list of subq per dataset.
    - For x_col 'n_subquantizers', num_subq_plot_subquantizers restricts which subq (x-axis points) are shown: same format.
    - When x_col is 'bits_per_vector': grouping is controlled by BITS_PER_VECTOR_GROUPING config
      (can be "nbits" or "n_subquantizers").
    
    Args:
        df: DataFrame with results
        x_col: Column name to plot on x-axis
        x_label: Display label for x-axis
        y_col: Column name to plot on y-axis (default: "rel_error_mean")
        y_label: Display label for y-axis (if None, auto-generated from y_col)
        methods: List of methods to plot (if None, uses all available)
        datasets: List of datasets to plot (if None, uses all available)
        ylim: Optional tuple (ymin, ymax) to set y-axis limits (if None, uses auto)
        group_by: Optional override for grouping ("nbits" or "n_subquantizers"). If None, uses default logic.
        output_dir: Directory to save plots
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())
    
    if output_dir is None:
        output_dir = Path("./../../experiments/plots/relerr_cpp")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Determine grouping variable based on x_col (or use group_by if provided)
    if group_by is not None:
        # Override default grouping with explicit group_by parameter.
        group_aliases = {
            "nbits": "nbits",
            "bits": "nbits",
            "n_subquantizers": "n_subquantizers",
            "nsubq": "n_subquantizers",
            "subq": "n_subquantizers",
            "M": "n_subquantizers",
        }
        group_col = group_aliases.get(group_by)
        if group_col is None:
            raise ValueError(f"group_by must be 'nbits' or 'n_subquantizers', got {group_by}")
        group_label_prefix = "nbits=" if group_col == "nbits" else "subq="
    elif x_col == "n_subquantizers":
        group_col = "nbits"
        group_label_prefix = "nbits="
    elif x_col == "nbits":
        group_col = "n_subquantizers"
        group_label_prefix = "subq="
    elif x_col == "bits_per_vector":
        # Use configuration to determine grouping
        if BITS_PER_VECTOR_GROUPING == "nbits":
            group_col = "nbits"
            group_label_prefix = "nbits="
        else:  # default to n_subquantizers
            group_col = "n_subquantizers"
            group_label_prefix = "subq="
    else:
        # For other columns, default to grouping by nbits
        group_col = "nbits"
        group_label_prefix = "nbits="
    
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].copy()
            if sub.empty:
                continue
            # For nbits x-axis: optionally restrict which n_subquantizers (curves) to show (per dataset or global)
            if x_col == "nbits" and nbits_subquantizers is not None:
                subq_list = nbits_subquantizers.get(dataset) if isinstance(nbits_subquantizers, dict) else nbits_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            # For n_subquantizers x-axis: optionally restrict which subq (x-axis points) to show (per dataset or global)
            if x_col == "n_subquantizers" and num_subq_plot_subquantizers is not None:
                subq_list = num_subq_plot_subquantizers.get(dataset) if isinstance(num_subq_plot_subquantizers, dict) else num_subq_plot_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            if sub.empty:
                continue

            # Create dynamic color and marker maps based on actual data values
            unique_group_values = sorted(sub[group_col].unique())
            color_map = create_dynamic_color_map(unique_group_values)
            marker_map = create_dynamic_marker_map(unique_group_values)
            
            # Create figure (using default size from rcParams)
            fig, ax = plt.subplots()
            
            # Group by the grouping column and plot each group
            grouped = sub.groupby(group_col)
            for group_val, group_df in grouped:
                # Sort by x_col for proper line plotting
                group_df_sorted = group_df.sort_values(x_col)
                
                # Get color and marker for this group
                color = color_map[group_val]
                marker = marker_map[group_val]
                
                # Plot line without error bars
                ax.plot(
                    group_df_sorted[x_col],
                    group_df_sorted[y_col],
                    label=f"{group_label_prefix}{group_val}",
                    color=color,
                    marker=marker,
                    markersize=12,
                    linewidth=2,
                    markeredgewidth=2,
                    markeredgecolor="black",
                )
            
            ax.set_xlabel(x_label, fontsize=40)
            # Set y-axis label - use provided y_label or generate from y_col
            if y_label is None:
                if y_col == "rel_error_mean":
                    y_label = "Avg Relative Error"
                elif y_col == "adc_time_s":
                    y_label = "ADC time (seconds)"
                elif y_col == "train_time_s":
                    y_label = "Training time (seconds)"
                elif y_col == "distance_table_time_s":
                    y_label = "Distance table time (seconds)"
                elif y_col == "encoding_time_s":
                    y_label = "Encoding time (seconds)"
                    y_label = y_col.replace("_", " ").title()
            ax.set_ylabel(y_label, fontsize=40)
            # No title - method and dataset are specified as inputs
            ax.tick_params(labelsize=40)
            if ADC_TIME_UNIT == "s":
                ax.ticklabel_format(style="scientific", axis="both", scilimits=(-3, 3))
                for axis in (ax.xaxis, ax.yaxis):
                    axis.offsetText.set_fontsize(28)
                ax.xaxis.offsetText.set_position((1.05, 0))
            
            # Set y-axis limits if specified
            if ylim is not None:
                ax.set_ylim(ylim)
            # Use log scale for subquantizers (like reference image)
            # You can add other columns here if you want log scale
            if x_col == "n_subquantizers":
                ax.set_xscale("log")
                unique_subq = sorted(sub[x_col].unique())
                if len(unique_subq) > 4:
                    indices = np.linspace(0, len(unique_subq) - 1, 4, dtype=int)
                    tick_values = [unique_subq[i] for i in indices]
                else:
                    tick_values = unique_subq
                # Store tick values to reapply after tight_layout
                tick_values_to_use = tick_values
            elif x_col == "nbits":
                # Explicitly show every unique nbits value on the x-axis
                unique_nbits = sorted(sub[x_col].unique())
                tick_values_to_use = unique_nbits
                ax.set_xticks(unique_nbits)
            
            # Grid styling like reference
            ax.grid(alpha=0.8, axis='y', linestyle='--')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            # Place legend outside plot when nbits or bits_per_vector is x-axis to avoid interference
            if x_col in ["nbits", "bits_per_vector"]:
                ax.legend(frameon=False, loc='center left', bbox_to_anchor=(1.05, 0.5))
            else:
                ax.legend(frameon=False, loc='best')
            
            # Apply tight_layout first
            plt.tight_layout()
            
            # Re-apply ticks for log scale after tight_layout to ensure they're preserved
            if x_col == "n_subquantizers" and tick_values_to_use is not None:
                ax.set_xticks(tick_values_to_use)
                ax.set_xticklabels([int(x) for x in tick_values_to_use])
            
            # Save figure with consistent DPI
            safe_method = method.lower()
            safe_dataset = dataset.lower()
            safe_x = x_col.replace("_", "")
            # Include y_col in filename if it's not the default
            if y_col != "rel_error_mean":
                safe_y = y_col.replace("_", "")
                savepath = output_dir / f"{safe_y}_{safe_method}_{safe_dataset}_{safe_x}.pdf"
            else:
                savepath = output_dir / f"relerr_{safe_method}_{safe_dataset}_{safe_x}.pdf"
            # fig.savefig(savepath, bbox_inches='tight', dpi=300)
            fig.savefig(savepath, dpi=300)
            
            print(f"Saved figure to {savepath}")
            
            plt.show()
            plt.close(fig)


def plot_bar_chart(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    y_col: str = "train_time_s",
    y_label: str = None,
    methods: list = None,
    datasets: list = None,
    group_by: str = None,
    output_dir: Path = None
):
    """
    Plot bar chart (bin-style) for y_col vs x_col, with separate figures per method and dataset.
    
    Args:
        df: DataFrame with results
        x_col: Column name to plot on x-axis (will be grouped into bars)
        x_label: Display label for x-axis
        y_col: Column name to plot on y-axis (default: "train_time_s")
        y_label: Display label for y-axis (if None, auto-generated from y_col)
        methods: List of methods to plot (if None, uses all available)
        datasets: List of datasets to plot (if None, uses all available)
        group_by: "nbits" or "n_subquantizers" - how to group the bars (different bars for each group)
        output_dir: Directory to save plots
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())
    
    if output_dir is None:
        output_dir = Path("./../../experiments/plots/relerr_cpp")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Determine grouping variable
    if group_by is None:
        # Default: group by the other variable (if x_col is nbits, group by n_subquantizers)
        if x_col == "nbits":
            group_col = "n_subquantizers"
            group_label_prefix = "subq="
        elif x_col == "n_subquantizers":
            group_col = "nbits"
            group_label_prefix = "nbits="
        else:
            group_col = "nbits"
            group_label_prefix = "nbits="
    elif group_by == "nbits":
        group_col = "nbits"
        group_label_prefix = "nbits="
    elif group_by == "n_subquantizers":
        group_col = "n_subquantizers"
        group_label_prefix = "subq="
    else:
        raise ValueError(f"group_by must be 'nbits' or 'n_subquantizers', got {group_by}")
    
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].copy()
            if sub.empty:
                continue
            
            # Create figure
            fig, ax = plt.subplots()
            
            # Get unique x_col and group_col values
            unique_x_vals = sorted(sub[x_col].unique())
            unique_group_vals = sorted(sub[group_col].unique())
            
            # Number of groups (x_col values) and bars per group (group_col values)
            n_groups = len(unique_x_vals)
            n_bars_per_group = len(unique_group_vals)
            
            # Set up bar positions for grouped bars
            bar_width = 0.8 / n_bars_per_group
            x_positions = np.arange(n_groups)
            
            # Color palette for different group_col values - same color for same group_col value
            colors = plt.cm.tab10(np.linspace(0, 1, n_bars_per_group))
            
            # Plot bars for each group_col value
            for i, group_val in enumerate(unique_group_vals):
                # Calculate y values for this group_col value across all x_col values
                y_vals = []
                for x_val in unique_x_vals:
                    # Get data for this specific x_col and group_col combination
                    data = sub[(sub[x_col] == x_val) & (sub[group_col] == group_val)]
                    if not data.empty:
                        # Use the actual value (not averaged) - if multiple rows exist, take mean
                        y_val = data[y_col].iloc[0] if len(data) == 1 else data[y_col].mean()
                        y_vals.append(y_val)
                        y_vals.append(0)
                
                # Calculate bar positions (offset for grouped bars)
                bar_positions = x_positions + i * bar_width
                
                # Plot bars - all bars with same group_col value get the same color
                ax.bar(
                    bar_positions,
                    y_vals,
                    width=bar_width,
                    label=f"{group_label_prefix}{group_val}",
                    color=colors[i],
                    edgecolor="black",
                    linewidth=2,
                    alpha=0.8
                )
            
            # Set labels and styling
            ax.set_xlabel(x_label, fontsize=40)
            if y_label is None:
                if y_col == "train_time_s":
                    y_label = "Training time (seconds)"
                elif y_col == "adc_time_s":
                    y_label = "ADC time (seconds)"
                elif y_col == "distance_table_time_s":
                    y_label = "Distance table time (seconds)"
                    y_label = y_col.replace("_", " ").title()
            ax.set_ylabel(y_label, fontsize=40)
            ax.tick_params(labelsize=40)
            
            # Set x-axis ticks at the center of each group
            ax.set_xticks(x_positions)
            ax.set_xticklabels([int(x) if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer()) else x 
                               for x in unique_x_vals])
            
            # Add legend
            # ax.legend(frameon=False, fontsize=30)
            
            # Grid styling - match reference image style
            ax.grid(alpha=0.8, axis='y', linestyle='--')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            # Apply tight_layout
            plt.tight_layout()
            
            # Save figure
            safe_method = method.lower()
            safe_dataset = dataset.lower()
            safe_x = x_col.replace("_", "")
            safe_y = y_col.replace("_", "")
            savepath = output_dir / f"bar_{safe_y}_{safe_method}_{safe_dataset}_{safe_x}.pdf"
            # fig.savefig(savepath, bbox_inches='tight', dpi=300)
            fig.savefig(savepath, dpi=300)
            print(f"Saved figure to {savepath}")
            
            plt.show()
            plt.close(fig)


def plot_compression_rate_vs_y(
    df: pd.DataFrame,
    y_col: str,
    y_label: str,
    methods: list = None,
    datasets: list = None,
    group_by: str = "nbits",
    output_dir: Path = None,
    nbits_subquantizers=None,
    num_subq_plot_subquantizers=None,
):
    """
    Plot chosen metric vs compression rate, with separate figures per method and dataset.
    X-axis: compression_rate, Y-axis: y_col.
    Curves are grouped by group_by ("nbits" or "n_subquantizers").
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())

    if output_dir is None:
        output_dir = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if y_col not in df.columns:
        return
    group_col = "nbits" if group_by == "nbits" else "n_subquantizers"
    group_label_prefix = "nbits=" if group_by == "nbits" else "subq="
    safe_y = y_col.replace("_", "")

    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].copy()
            if sub.empty:
                continue
            sub = sub.dropna(subset=["compression_rate", y_col])
            if sub.empty:
                continue

            if nbits_subquantizers is not None:
                subq_list = nbits_subquantizers.get(dataset) if isinstance(nbits_subquantizers, dict) else nbits_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            if num_subq_plot_subquantizers is not None:
                subq_list = num_subq_plot_subquantizers.get(dataset) if isinstance(num_subq_plot_subquantizers, dict) else num_subq_plot_subquantizers
                if subq_list is not None:
                    sub = sub[sub["n_subquantizers"].isin(subq_list)]
            if sub.empty:
                continue

            unique_group_values = sorted(sub[group_col].unique())
            color_map = create_dynamic_color_map(unique_group_values)
            marker_map = create_dynamic_marker_map(unique_group_values)

            fig, ax = plt.subplots()
            grouped = sub.groupby(group_col)
            for group_val, group_df in grouped:
                group_df_sorted = group_df.sort_values("compression_rate")
                color = color_map[group_val]
                marker = marker_map[group_val]
                ax.plot(
                    group_df_sorted["compression_rate"],
                    group_df_sorted[y_col],
                    label=f"{group_label_prefix}{group_val}",
                    color=color,
                    marker=marker,
                    markersize=12,
                    linewidth=2,
                    markeredgewidth=2,
                    markeredgecolor="black",
                )

            ax.set_xlabel("Compression rate", fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            ax.tick_params(labelsize=40)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.grid(alpha=0.8, axis="y", linestyle="--")
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.05, 0.5))
            plt.tight_layout()

            safe_method = method.lower()
            safe_dataset = dataset.lower()
            savepath = output_dir / f"compression_rate_vs_{safe_y}_{safe_method}_{safe_dataset}.pdf"
            fig.savefig(savepath, dpi=300)
            print(f"Saved figure to {savepath}")
            plt.show()
            plt.close(fig)


def plot_opq_vs_pq_rot_percent(
    df: pd.DataFrame,
    dataset: str,
    nbits_list,
    n_subquantizers_list,
    y_col: str = "rel_error_mean",
    y_label: str = None,
    higher_is_better: bool = False,
    output_dir: Path = None,
    figsize: tuple = (10, 6),
    n_bins: int = 20,
):
    """
    Plot y_col vs % of (max_opq_rot_train_samples / rot_train_sz), averaged across
    all (nbits, n_subquantizers) in nbits_list x n_subquantizers_list.
    PQ: horizontal line = mean y_col over those pairs.
    OPQ: curve = for each x (pct_rot bin), mean y_col over all selected pairs in that bin.
    Plot PQ and OPQ as lines without shaded comparison bands.
    nbits_list and n_subquantizers_list can be lists or single ints (converted to list).
    """
    if output_dir is None:
        output_dir = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    nbits_list = [nbits_list] if np.isscalar(nbits_list) else list(nbits_list)
    n_subquantizers_list = [n_subquantizers_list] if np.isscalar(n_subquantizers_list) else list(n_subquantizers_list)

    for col in ["rot_train_sz", "max_opq_rot_train_samples"]:
        if col not in df.columns:
            raise ValueError(
                f"DataFrame must contain OPQ columns 'rot_train_sz' and 'max_opq_rot_train_samples'. Missing: {col}"
            )

    # PQ: average rel_error over all (dataset, nbits, n_subquantizers) in the lists
    pq = df[(df["method"] == "PQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))]
    if pq.empty:
        raise ValueError(f"No PQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    if y_col not in pq.columns:
        raise ValueError(f"Column '{y_col}' not found in dataframe")
    pq_val = float(pq[y_col].mean())

    # OPQ: all rows for (dataset, nbits, n_subquantizers) in the lists; compute pct_rot
    opq = df[(df["method"] == "OPQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))].copy()
    opq = opq.dropna(subset=["rot_train_sz", "max_opq_rot_train_samples", y_col])
    if opq.empty:
        raise ValueError(f"No OPQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    opq["pct_rot"] = 100.0 * opq["max_opq_rot_train_samples"] / opq["rot_train_sz"]

    # Bin pct_rot and average y_col in each bin (one curve across all (nbits, M))
    pct_min, pct_max = opq["pct_rot"].min(), opq["pct_rot"].max()
    if pct_min >= pct_max or np.isclose(pct_min, pct_max):
        # All OPQ runs used same pct (e.g. 100%); single point.
        x = np.array([pct_min])
        y_opq = np.array([opq[y_col].mean()])
    else:
        bins = np.linspace(pct_min, pct_max, n_bins + 1)
        opq["_bin"] = pd.cut(opq["pct_rot"], bins=bins, include_lowest=True)
        agg = opq.groupby("_bin", observed=True).agg({"pct_rot": "mean", y_col: "mean"}).reset_index()
        agg = agg.dropna(subset=["pct_rot", y_col])
        if agg.empty:
            raise ValueError(f"No OPQ bins for dataset={dataset}, metric={y_col} after pct_rot binning")
        x = agg["pct_rot"].values
        y_opq = agg[y_col].values
    y_pq = np.full_like(x, pq_val)

    fig, ax = plt.subplots(figsize=figsize)
    # Custom markers and colors from notebook palette (consistent with other figures)
    ax.plot(x, y_pq, color=COLOR_PALETTE[0], linewidth=2, marker="o", markersize=12, markeredgecolor="black", markeredgewidth=2, label="PQ")
    ax.plot(x, y_opq, color=COLOR_PALETTE[1], linewidth=2, marker="s", markersize=12, markeredgecolor="black", markeredgewidth=2, label="OPQ")
    ax.set_xticks(x)
    ax.set_xlabel("% data used to learn R", fontsize=40)
    if y_label is None:
        y_label = y_col.replace("_", " ").title()
    ax.set_ylabel(y_label, fontsize=40)
    ax.tick_params(labelsize=40)
    # Widen y-axis for a better view (add ~15% padding above and below data range)
    y_lo = min(pq_val, y_opq.min())
    y_hi = max(pq_val, y_opq.max())
    pad = max((y_hi - y_lo) * 0.15, 0.005)
    ax.set_ylim(y_lo - pad, y_hi + pad)
    ax.grid(alpha=0.8, axis="y", linestyle="--")
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Keep PQ/OPQ legend horizontal and out of the data region.
    ax.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        ncol=2,
        fontsize=21,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    suffix = f"M{min(n_subquantizers_list)}-{max(n_subquantizers_list)}_nbits{min(nbits_list)}-{max(nbits_list)}" if (len(n_subquantizers_list) > 1 or len(nbits_list) > 1) else f"M{n_subquantizers_list[0]}_nbits{nbits_list[0]}"
    safe_y = y_col.replace("_", "")
    savepath = output_dir / f"opq_vs_pq_rot_pct_{safe_y}_{dataset}_{suffix}.pdf"
    fig.savefig(savepath, dpi=300)
    print(f"Saved {savepath}")
    plt.show()
    plt.close(fig)


def plot_opq_vs_pq_train_time_rot_percent(
    df: pd.DataFrame,
    dataset: str,
    nbits_list,
    n_subquantizers_list,
    output_dir: Path = None,
    figsize: tuple = (10, 6),
    n_bins: int = 20,
):
    """
    Plot training time vs % of data used to learn R, averaged across
    all (nbits, n_subquantizers) in nbits_list x n_subquantizers_list.
    PQ: horizontal line = mean train_time_s over those pairs.
    OPQ: curve = train_time_s + opq_train_time_s per run, binned by pct_rot and averaged.
    Same x-axis and averaging logic as plot_opq_vs_pq_rot_percent.
    """
    if output_dir is None:
        output_dir = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    nbits_list = [nbits_list] if np.isscalar(nbits_list) else list(nbits_list)
    n_subquantizers_list = [n_subquantizers_list] if np.isscalar(n_subquantizers_list) else list(n_subquantizers_list)

    for col in ["rot_train_sz", "max_opq_rot_train_samples", "train_time_s"]:
        if col not in df.columns:
            raise ValueError(f"DataFrame must contain '{col}'. Missing: {col}")

    # PQ: average train_time_s over all (dataset, nbits, n_subquantizers) in the lists
    pq = df[(df["method"] == "PQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))]
    if pq.empty:
        raise ValueError(f"No PQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    pq_time = float(pq["train_time_s"].mean())

    # OPQ: train_time_s + opq_train_time_s (use train_time_s only if opq_train_time_s missing)
    opq = df[(df["method"] == "OPQ") & (df["dataset"] == dataset) & (df["nbits"].isin(nbits_list)) & (df["n_subquantizers"].isin(n_subquantizers_list))].copy()
    opq = opq.dropna(subset=["rot_train_sz", "max_opq_rot_train_samples", "train_time_s"])
    if opq.empty:
        raise ValueError(f"No OPQ data for dataset={dataset}, nbits in {nbits_list}, n_subquantizers in {n_subquantizers_list}")
    if "opq_train_time_s" in opq.columns:
        opq["total_train_time_s"] = opq["train_time_s"] + opq["opq_train_time_s"].fillna(0)
    else:
        opq["total_train_time_s"] = opq["train_time_s"]
    opq["pct_rot"] = 100.0 * opq["max_opq_rot_train_samples"] / opq["rot_train_sz"]

    # Bin pct_rot and average total_train_time_s in each bin.
    pct_min, pct_max = opq["pct_rot"].min(), opq["pct_rot"].max()
    if pct_min >= pct_max or np.isclose(pct_min, pct_max):
        x = np.array([pct_min])
        y_opq = np.array([opq["total_train_time_s"].mean() / 60])
    else:
        bins = np.linspace(pct_min, pct_max, n_bins + 1)
        opq["_bin"] = pd.cut(opq["pct_rot"], bins=bins, include_lowest=True)
        agg = opq.groupby("_bin", observed=True).agg({"pct_rot": "mean", "total_train_time_s": "mean"}).reset_index()
        agg = agg.dropna(subset=["pct_rot", "total_train_time_s"])
        if agg.empty:
            raise ValueError(f"No OPQ train-time bins for dataset={dataset} after pct_rot binning")
        x = agg["pct_rot"].values
        y_opq = agg["total_train_time_s"].values / 60
    pq_time = pq_time / 60  # convert to minutes
    y_pq = np.full_like(x, pq_time)

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(x, y_pq, color=COLOR_PALETTE[0], linewidth=2, marker="o", markersize=12, markeredgecolor="black", markeredgewidth=2, label="PQ")
    ax.plot(x, y_opq, color=COLOR_PALETTE[1], linewidth=2, marker="s", markersize=12, markeredgecolor="black", markeredgewidth=2, label="OPQ")
    ax.set_xticks(x)
    ax.set_xlabel("% data used to learn R", fontsize=40)
    ax.set_ylabel("Train time (m)", fontsize=40)
    ax.tick_params(labelsize=40)
    y_lo = min(pq_time, y_opq.min())
    y_hi = max(pq_time, y_opq.max())
    pad = max((y_hi - y_lo) * 0.15, 0.05)
    ax.set_ylim(y_lo - pad, y_hi + pad)
    # Exactly 3 y-ticks: beginning, middle, end
    ax.set_yticks([y_lo, (y_lo + y_hi) / 2, y_hi])
    ax.grid(alpha=0.8, axis="y", linestyle="--")
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Keep PQ/OPQ legend horizontal and out of the data region.
    ax.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        ncol=2,
        fontsize=21,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    suffix = f"M{min(n_subquantizers_list)}-{max(n_subquantizers_list)}_nbits{min(nbits_list)}-{max(nbits_list)}" if (len(n_subquantizers_list) > 1 or len(nbits_list) > 1) else f"M{n_subquantizers_list[0]}_nbits{nbits_list[0]}"
    savepath = output_dir / f"opq_vs_pq_train_time_rot_pct_{dataset}_{suffix}.pdf"
    fig.savefig(savepath, dpi=300)
    print(f"Saved {savepath}")
    plt.show()
    plt.close(fig)


# Generate plots based on configuration (for each Y_METRIC: relerr vs nbits, vs n_subq, etc.)
metrics_to_plot = []
for m in Y_METRICS:
    if m == "recall":
        metrics_to_plot.extend(["recall_1", "recall_10", "recall_100"])
        metrics_to_plot.append(m)

for x_col, x_label in X_COLUMNS_TO_PLOT:
    if x_col not in plot_df.columns:
        print(f"Warning: Column '{x_col}' not found in dataframe. Skipping...")
        continue

    ylim = YLIM_CONFIG.get(x_col, None)

    for metric in metrics_to_plot:
        if metric not in Y_METRIC_MAP:
            continue
        y_col, y_label = Y_METRIC_MAP[metric]
        if y_col not in plot_df.columns:
            continue

        print(f"\n{'='*60}")
        print(f"Plotting: {y_label} vs {x_label}")
        print(f"Methods: {METHODS_TO_PLOT}, Datasets: {DATASETS_TO_PLOT}")
        if ylim is not None:
            print(f"Y-axis limits: {ylim}")
        print(f"{'='*60}\n")
        plot_relerr_vs_x(
            plot_df,
            x_col,
            x_label,
            y_col=y_col,
            y_label=y_label,
            methods=METHODS_TO_PLOT,
            datasets=DATASETS_TO_PLOT,
            ylim=ylim,
            output_dir=Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures"),
            nbits_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
            num_subq_plot_subquantizers=NUM_SUBQ_PLOT_SUBQUANTIZERS,
        )

# Generate additional plots with different y-axis columns
for x_col, x_label, y_col, y_label, group_by in ADDITIONAL_PLOTS:
    # Check if columns exist in dataframe
    if x_col not in plot_df.columns:
        print(f"Warning: Column '{x_col}' not found in dataframe. Skipping...")
        continue
    if y_col not in plot_df.columns:
        print(f"Warning: Column '{y_col}' not found in dataframe. Skipping...")
        continue

    # Get ylim for this x-axis column if specified
    ylim = YLIM_CONFIG.get(x_col, None)

    print(f"\n{'='*60}")
    print(f"Plotting: {y_label} vs {x_label} (grouped by {group_by})")
    print(f"Methods: {METHODS_TO_PLOT}, Datasets: {DATASETS_TO_PLOT}")
    if ylim is not None:
        print(f"Y-axis limits: {ylim}")
    print(f"{'='*60}\n")
    plot_relerr_vs_x(
        plot_df,
        x_col,
        x_label,
        y_col=y_col,
        y_label=y_label,
        methods=METHODS_TO_PLOT,
        datasets=DATASETS_TO_PLOT,
        ylim=ylim,
        group_by=group_by,
        output_dir=Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures"),
        nbits_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
        num_subq_plot_subquantizers=NUM_SUBQ_PLOT_SUBQUANTIZERS,
    )

# Generate compression rate vs Y-axis plots (relerr, recall, spearman, recon_error, etc.)
# Compression rate vs Y (for metrics in Y_METRICS)
if "compression_rate" in plot_df.columns:
    for metric in metrics_to_plot:
        if metric not in Y_METRIC_MAP:
            continue
        y_col, y_label = Y_METRIC_MAP[metric]
        print("\n" + "=" * 60)
        print(f"Plotting: Compression rate vs {y_label}")
        print(f"Methods: {METHODS_TO_PLOT}, Datasets: {DATASETS_TO_PLOT}")
        print("=" * 60 + "\n")
        plot_compression_rate_vs_y(
            plot_df,
            y_col=y_col,
            y_label=y_label,
            methods=METHODS_TO_PLOT,
            datasets=DATASETS_TO_PLOT,
            group_by=COMPRESSION_RATE_GROUPING,
            output_dir=Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures"),
            nbits_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
            num_subq_plot_subquantizers=NUM_SUBQ_PLOT_SUBQUANTIZERS,
        )







In [ ]:
# Generate bar plots (bin-style plots) for training time
for x_col, x_label, y_col, y_label, average_over in BAR_PLOTS:
    # Check if columns exist in dataframe
    if x_col not in plot_df.columns:
        print(f"Warning: Column '{x_col}' not found in dataframe. Skipping...")
        continue
    if y_col not in plot_df.columns:
        print(f"Warning: Column '{y_col}' not found in dataframe. Skipping...")
        continue
    
    print(f"\n{'='*60}")
    print(f"Plotting bar chart: {y_label} vs {x_label} (averaged over {average_over})")
    print(f"Methods: {METHODS_TO_PLOT}, Datasets: {DATASETS_TO_PLOT}")
    print(f"{'='*60}\n")
    plot_bar_chart(
        plot_df, 
        x_col, 
        x_label,
        y_col=y_col,
        y_label=y_label,
        methods=METHODS_TO_PLOT,
        datasets=DATASETS_TO_PLOT,
        group_by=average_over,
        output_dir=Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")
    )

In [ ]:
# OPQ vs PQ: Y_METRICS vs % of rot_train used, averaged across (nbits, n_subquantizers) in the lists
# Requires OPQ results with columns rot_train_sz and max_opq_rot_train_samples (e.g. from train_opq outputs).
OPQ_VS_PQ_REQUESTED_DATASETS = ["deep", "bigann", "gist", "msmarco", "openai"]
OPQ_VS_PQ_NBITS = [4, 6, 8, 10, 12]   # list of nbits to average over
# Dataset-specific n_subquantizers (M). Use list for same M for all, or dict for per-dataset.
OPQ_VS_PQ_M = {
    "deep": [1, 4, 12, 24, 32, 96],
    "bigann": [1, 4, 16, 32, 64, 128],
    "gist": [1, 8, 40, 60, 320, 960],
    "msmarco": [1, 8, 32, 64, 256, 1024],
    "openai": [1, 32, 128, 256, 512, 1536],
}

# Optional: use jz_relerr2 for OPQ (OPQ from jz_relerr2, PQ from relerr_cpp). None = use main plot_df.
OPQ_VS_PQ_OPQ_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/jz_relerr3")  # or None for relerr_cpp
OPQ_VS_PQ_PQ_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp")

if OPQ_VS_PQ_OPQ_DIR is not None:
    relerr_opq = load_relerr_data(OPQ_VS_PQ_OPQ_DIR)
    relerr_opq = relerr_opq[relerr_opq["method"] == "OPQ"]
    relerr_pq = load_relerr_data(OPQ_VS_PQ_PQ_DIR)
    relerr_pq = relerr_pq[relerr_pq["method"] == "PQ"]
    opq_vs_pq_plot_df = build_plot_df(pd.concat([relerr_opq, relerr_pq], ignore_index=True), OPQ_VS_PQ_PQ_DIR)
    opq_vs_pq_output_dir = OPQ_VS_PQ_OPQ_DIR / "figures"
else:
    opq_vs_pq_plot_df = plot_df
    opq_vs_pq_output_dir = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")

HIGHER_IS_BETTER_METRICS = {"spearman", "recall_1", "recall_10", "recall_100"}

# Produce comparison figures only where both methods are present.
_available_methods_by_dataset = (
    opq_vs_pq_plot_df.groupby("dataset")["method"]
    .apply(lambda s: set(s.dropna()))
    .to_dict()
)
OPQ_VS_PQ_DATASETS = [
    dataset
    for dataset in OPQ_VS_PQ_REQUESTED_DATASETS
    if {"PQ", "OPQ"}.issubset(_available_methods_by_dataset.get(dataset, set()))
]
_skipped_opq_vs_pq = [dataset for dataset in OPQ_VS_PQ_REQUESTED_DATASETS if dataset not in OPQ_VS_PQ_DATASETS]
if _skipped_opq_vs_pq:
    print(f"Skipping OPQ vs PQ datasets without both PQ and OPQ rows: {_skipped_opq_vs_pq}")
print(f"OPQ vs PQ datasets to plot: {OPQ_VS_PQ_DATASETS}")

for dataset in OPQ_VS_PQ_DATASETS:
    m_list = OPQ_VS_PQ_M[dataset] if isinstance(OPQ_VS_PQ_M, dict) else OPQ_VS_PQ_M
    for metric in metrics_to_plot:
        if metric not in Y_METRIC_MAP:
            continue
        y_col, y_label = Y_METRIC_MAP[metric]
        if y_col not in opq_vs_pq_plot_df.columns:
            continue
        try:
            plot_opq_vs_pq_rot_percent(
                opq_vs_pq_plot_df,
                dataset=dataset,
                nbits_list=OPQ_VS_PQ_NBITS,
                n_subquantizers_list=m_list,
                y_col=y_col,
                y_label=y_label,
                higher_is_better=metric in HIGHER_IS_BETTER_METRICS,
                output_dir=opq_vs_pq_output_dir,
                figsize=(10, 6),
                n_bins=20,
            )
        except (ValueError, KeyError) as e:
            print(f"Skipping OPQ vs PQ for {dataset} {metric}: {e}")


In [ ]:
# OPQ vs PQ: training time vs % of data used to learn R (same lists as above, separate function)
# PQ = train_time_s; OPQ = train_time_s + opq_train_time_s; averaged over OPQ_VS_PQ_NBITS and OPQ_VS_PQ_M
for dataset in OPQ_VS_PQ_DATASETS:
    m_list = OPQ_VS_PQ_M[dataset] if isinstance(OPQ_VS_PQ_M, dict) else OPQ_VS_PQ_M
    try:
        plot_opq_vs_pq_train_time_rot_percent(
            opq_vs_pq_plot_df,
            dataset=dataset,
            nbits_list=OPQ_VS_PQ_NBITS,
            n_subquantizers_list=m_list,
            output_dir=opq_vs_pq_output_dir,
            figsize=(10, 6),
            n_bins=20,
        )
    except (ValueError, KeyError) as e:
        print(f"Skipping OPQ vs PQ train time for {dataset}: {e}")

In [ ]:
# =============================================================================
# Train size experiments: train_size vs relative error (from /mnthdd/cpanourg/2-hdvc/results/urania_results/results/train_size_exps)
# Legends grouped by bits_per_vector (nbits × n_subquantizers)
# =============================================================================

TRAIN_SIZE_EXPS_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/train_size_exps")
TRAIN_SIZE_FIGURES_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/train_size_exps/figures")

# Optional: filter which bits_per_vector to plot. None = all; e.g. [64, 96] = only those
BITS_PER_VECTOR_FILTER = None  # or [64], [96], [64, 96], etc.

# Output format: "svg" or "pdf"
TRAIN_SIZE_FIGURES_FORMAT = "svg"

train_size_df = load_relerr_data(TRAIN_SIZE_EXPS_DIR)
train_size_df = train_size_df[train_size_df["method"] == "PQ"]  # Only PQ in train_size_exps

methods_ts = sorted(train_size_df["method"].unique())
datasets_ts = sorted(train_size_df["dataset"].unique())
TRAIN_SIZE_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for method in methods_ts:
    for dataset in datasets_ts:
        sub = train_size_df[(train_size_df["method"] == method) & (train_size_df["dataset"] == dataset)]
        if sub.empty:
            continue
        if "train_size" not in sub.columns or "rel_error_mean" not in sub.columns or "nb" not in sub.columns:
            continue

        # Convert train_size to percentage of database (nb)
        sub = sub.copy()
        sub["train_pct"] = sub["train_size"] / sub["nb"] * 100

        fig, ax = plt.subplots(figsize=(10, 6))
        unique_bpv = sorted(sub["bits_per_vector"].unique())
        if BITS_PER_VECTOR_FILTER is not None:
            unique_bpv = [b for b in unique_bpv if b in BITS_PER_VECTOR_FILTER]
        if not unique_bpv:
            continue
        color_map = create_dynamic_color_map(unique_bpv)
        marker_map = create_dynamic_marker_map(unique_bpv)

        for bpv in unique_bpv:
            grp = sub[sub["bits_per_vector"] == bpv].sort_values("train_pct")
            unique_nbits = sorted(grp["nbits"].unique())
            if len(unique_nbits) == 1:
                nbits_label = f"nbits={int(unique_nbits[0])}"
            elif len(unique_nbits) <= 4:
                nbits_label = "nbits=" + ",".join(str(int(v)) for v in unique_nbits)
            else:
                nbits_label = f"nbits={int(unique_nbits[0])}-{int(unique_nbits[-1])}"

            ax.plot(
                grp["train_pct"],
                grp["rel_error_mean"],
                color=color_map[bpv],
                marker=marker_map[bpv],
                markersize=12,
                linewidth=2,
                markeredgecolor="black",
                markeredgewidth=2,
                label=f"bits/vec={int(bpv)} ({nbits_label})",
            )
            # if "rel_error_std" in grp.columns:
            #     ax.fill_between(
            #         grp["train_size"],
            #         grp["rel_error_mean"] - grp["rel_error_std"],
            #         grp["rel_error_mean"] + grp["rel_error_std"],
            #         color=color_map[bpv],
            #         alpha=0.2,
            #     )

        sub_plotted = sub[sub["bits_per_vector"].isin(unique_bpv)]
        y_min = -0.1
        y_max = sub_plotted["rel_error_mean"].max() * 1.5
        ax.set_ylim(y_min, y_max)

        ax_label_fontsize = 40
        ax.set_xlabel("Train size (% DB)", fontsize=ax_label_fontsize)
        ax.set_ylabel("Avg Relative Error", fontsize=ax_label_fontsize)
        ax.tick_params(labelsize=ax_label_fontsize)
        # Use actual data points for x-ticks so they align under the points (e.g. gist has 2.5, 5, 12.5, 25, 37.5, 50)
        unique_pct = sorted(sub["train_pct"].unique())
        ax.set_xticks(unique_pct)
        ax.set_xticklabels([5, 10, 25, 50, 75, 100], rotation=90)
        ax.grid(alpha=0.8, axis="y", linestyle="--")
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=21)
        plt.tight_layout()
        savepath = TRAIN_SIZE_FIGURES_DIR / f"{dataset}_{method}_train_size_vs_relerr.{TRAIN_SIZE_FIGURES_FORMAT}"
        fig.savefig(savepath)
        print(f"Saved {savepath}")
        plt.show()
        plt.close(fig)

In [ ]:
# =============================================================================
# Averaged ADC CPU time per pair plots:
#   1) Mean adc_cpu_time_pp vs nbits (averaged over n_subquantizers)
#   2) Mean adc_cpu_time_pp vs n_subquantizers (averaged over nbits)
# One figure per (method, dataset) — same style as the rest of the notebook.
# =============================================================================

def plot_averaged_adc_time(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    avg_over: str,
    y_col: str = _adc_col,
    y_label: str = _adc_label,
    methods: list = None,
    datasets: list = None,
    output_dir: Path = None,
    allowed_subquantizers: dict = None,
):
    """Plot y_col averaged over `avg_over` for each unique value of `x_col`.

    For example, x_col="nbits", avg_over="n_subquantizers" produces one point
    per nbits value, each being the mean across all n_subquantizers experiments.

    allowed_subquantizers: dict mapping dataset -> list of n_subquantizers to keep.
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())
    if output_dir is None:
        output_dir = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")
    output_dir.mkdir(parents=True, exist_ok=True)

    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].copy()
            if sub.empty or y_col not in sub.columns:
                continue
            if allowed_subquantizers and dataset in allowed_subquantizers:
                sub = sub[sub["n_subquantizers"].isin(allowed_subquantizers[dataset])]
                if sub.empty:
                    continue

            agg = (
                sub.groupby(x_col)[y_col]
                .agg(["mean", "std"])
                .reset_index()
                .sort_values(x_col)
            )

            fig, ax = plt.subplots()
            ax.errorbar(
                agg[x_col], agg["mean"], yerr=agg["std"],
                fmt="o-", color=COLOR_PALETTE[0], markersize=12,
                linewidth=2, markeredgewidth=2, markeredgecolor="black",
                capsize=5, capthick=2, elinewidth=1.5,
            )

            ax.set_xlabel(x_label, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            ax.tick_params(labelsize=40)
            if ADC_TIME_UNIT == "s":
                ax.ticklabel_format(style="scientific", axis="both", scilimits=(-3, 3))
                for axis in (ax.xaxis, ax.yaxis):
                    axis.offsetText.set_fontsize(28)

            if x_col == "n_subquantizers":
                ax.set_xscale("log")
                unique_vals = sorted(agg[x_col].unique())
                if len(unique_vals) > 4:
                    idx = np.linspace(0, len(unique_vals) - 1, 4, dtype=int)
                    ax.set_xticks([unique_vals[i] for i in idx])
                    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
                else:
                    ax.set_xticks(unique_vals)
            else:
                ax.set_xticks(sorted(agg[x_col].unique()))

            ax.grid(alpha=0.8, axis="y", linestyle="--")
            for spine in ax.spines.values():
                spine.set_visible(False)

            plt.tight_layout()
            safe_y = y_col.replace("_", "")
            fname = f"avg_{safe_y}_vs_{x_col}_{method.lower()}_{dataset}.pdf"
            fig.savefig(output_dir / fname, dpi=300, bbox_inches="tight")
            print(f"Saved {output_dir / fname}")
            plt.show()
            plt.close(fig)


FIGURES_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")

# --- Plot 1: Mean ADC CPU time/pair vs nbits (averaged over n_subquantizers) ---
print("=" * 60)
print("Average adc_cpu_time_pp vs nbits (averaged over n_subquantizers)")
print("=" * 60)
plot_averaged_adc_time(
    plot_df,
    x_col="nbits",
    x_label="nbits",
    avg_over="n_subquantizers",
    methods=METHODS_TO_PLOT,
    datasets=DATASETS_TO_PLOT,
    output_dir=FIGURES_DIR,
    allowed_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
)

# --- Plot 2: Mean ADC CPU time/pair vs n_subquantizers (averaged over nbits) ---
print("=" * 60)
print("Average adc_cpu_time_pp vs n_subquantizers (averaged over nbits)")
print("=" * 60)
plot_averaged_adc_time(
    plot_df,
    x_col="n_subquantizers",
    x_label="Num subquantizers",
    avg_over="nbits",
    methods=METHODS_TO_PLOT,
    datasets=DATASETS_TO_PLOT,
    output_dir=FIGURES_DIR,
    allowed_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
)

In [ ]:
# =============================================================================
# Pareto plots: ADC CPU time per pair vs Avg Relative Error
#   1) One point per nbits   (averaged over all n_subquantizers)
#   2) One point per n_subquantizers (averaged over all nbits)
# Lower-left = better (faster AND more accurate).
# =============================================================================

def plot_pareto_adc_vs_relerr(
    df: pd.DataFrame,
    group_col: str,
    group_label: str,
    x_col: str = _adc_col,
    y_col: str = "rel_error_mean",
    x_label: str = _adc_label,
    y_label: str = "Avg Relative Error",
    methods: list = None,
    datasets: list = None,
    output_dir: Path = None,
    allowed_subquantizers: dict = None,
):
    """Pareto scatter: x = ADC time per pair, y = relative error.

    Each point is the mean of (x_col, y_col) across all experiments that
    share the same `group_col` value. Points are labelled and connected
    in sorted order of the grouping variable.

    allowed_subquantizers: dict mapping dataset -> list of n_subquantizers to keep.
    """
    if methods is None:
        methods = sorted(df["method"].unique())
    if datasets is None:
        datasets = sorted(df["dataset"].unique())
    if output_dir is None:
        output_dir = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")
    output_dir.mkdir(parents=True, exist_ok=True)

    for method in methods:
        for dataset in datasets:
            sub = df[
                (df["method"] == method) & (df["dataset"] == dataset)
            ].dropna(subset=[x_col, y_col])
            if sub.empty:
                continue
            if allowed_subquantizers and dataset in allowed_subquantizers:
                sub = sub[sub["n_subquantizers"].isin(allowed_subquantizers[dataset])]
                if sub.empty:
                    continue

            agg = (
                sub.groupby(group_col)[[x_col, y_col]]
                .mean()
                .reset_index()
                .sort_values(group_col)
            )
            if agg.empty:
                continue

            fig, ax = plt.subplots()

            # Connect points in order with a thin line
            ax.plot(
                agg[x_col], agg[y_col],
                "-", color="gray", linewidth=1, alpha=0.5, zorder=1,
            )

            # Scatter with per-value colours
            unique_vals = sorted(agg[group_col].unique())
            cmap = create_dynamic_color_map(unique_vals)
            mmap = create_dynamic_marker_map(unique_vals)

            for _, row in agg.iterrows():
                val = row[group_col]
                ax.scatter(
                    row[x_col], row[y_col],
                    color=cmap[val], marker=mmap[val],
                    s=200, edgecolors="black", linewidths=2, zorder=2,
                    label=f"{group_label}={int(val)}",
                )

            ax.set_xlabel(x_label, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            ax.tick_params(labelsize=40)
            if ADC_TIME_UNIT == "s":
                ax.ticklabel_format(style="scientific", axis="both", scilimits=(-3, 3))
                for axis in (ax.xaxis, ax.yaxis):
                    axis.offsetText.set_fontsize(28)
                ax.xaxis.offsetText.set_position((1.15, 0))
            ax.grid(alpha=0.8, axis="both", linestyle="--")
            for spine in ax.spines.values():
                spine.set_visible(False)

            ax.legend(
                frameon=False, loc="center left",
                bbox_to_anchor=(1.05, 0.5), fontsize=18,
            )
            plt.tight_layout()

            safe = f"pareto_{x_col}_vs_{y_col}_by_{group_col}_{method.lower()}_{dataset}"
            fig.savefig(output_dir / f"{safe}.pdf", dpi=300, bbox_inches="tight")
            print(f"Saved {output_dir / f'{safe}.pdf'}")
            plt.show()
            plt.close(fig)


FIGURES_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/relerr_cpp/figures")

# --- Pareto 1: grouped by nbits (averaged over n_subquantizers) ---
print("=" * 60)
print("Pareto: ADC CPU time/pair vs Avg Relative Error  (one point per nbits)")
print("=" * 60)
plot_pareto_adc_vs_relerr(
    plot_df,
    group_col="nbits",
    group_label="nbits",
    methods=METHODS_TO_PLOT,
    datasets=DATASETS_TO_PLOT,
    output_dir=FIGURES_DIR,
    allowed_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
)

# --- Pareto 2: grouped by n_subquantizers (averaged over nbits) ---
print("=" * 60)
print("Pareto: ADC CPU time/pair vs Avg Relative Error  (one point per n_subquantizers)")
print("=" * 60)
plot_pareto_adc_vs_relerr(
    plot_df,
    group_col="n_subquantizers",
    group_label="subq",
    methods=METHODS_TO_PLOT,
    datasets=DATASETS_TO_PLOT,
    output_dir=FIGURES_DIR,
    allowed_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
)

In [ ]:
# =============================================================================
# All-experiments ADC timing (from measure_all_adc.py)
# Plots ADC time vs nbits and vs n_subquantizers — one point per nbits / per M,
# aggregated over all experiments with that nbits / M.
# =============================================================================

ALL_ADC_BY_NBITS = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/all_adc_timing_by_nbits.csv")
ALL_ADC_BY_NSUBQ = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/all_adc_timing_by_nsubq.csv")
FIGURES_DIR_OPT = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/figures")

def _plot_all_adc_vs_param(path: Path, x_col: str, x_label: str, datasets: list = None):
    """Plot ADC time vs nbits or n_subquantizers from measure_all_adc.py output."""
    if not path.exists():
        print(f"⚠️  {path} not found — run measure_all_adc.py first")
        return
    df = pd.read_csv(path)
    if datasets is None:
        datasets = sorted(df["dataset"].unique())
    else:
        datasets = [d for d in datasets if d in df["dataset"].unique()]

    for dataset in datasets:
        sub = df[df["dataset"] == dataset].sort_values(x_col)
        if sub.empty:
            continue
        x_vals = sub[x_col].values
        y_vals = sub["adc_cpu_time_pp_mean"].values
        y_err = sub["adc_cpu_time_pp_std"].fillna(0).values

        fig, ax = plt.subplots()
        ax.errorbar(x_vals, y_vals, yerr=y_err, fmt="o-", markersize=12, linewidth=2,
                    markeredgewidth=2, markeredgecolor="black", capsize=5, elinewidth=1.5,
                    color="#2196F3")
        ax.set_xlabel(x_label, fontsize=40)
        ax.set_ylabel("ADC time (s)", fontsize=40)
        ax.tick_params(labelsize=40)
        ax.ticklabel_format(style="scientific", axis="y", scilimits=(-3, 3))
        ax.yaxis.offsetText.set_fontsize(28)
        ax.grid(alpha=0.8, axis="y", linestyle="--")
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_xticks(x_vals)
        ax.set_xticklabels([int(x) if x == int(x) else x for x in x_vals])
        plt.tight_layout()
        safe = f"all_adc_vs_{x_col}_pq_{dataset}"
        out = FIGURES_DIR_OPT / f"{safe}.pdf"
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print(f"Saved {out}")
        plt.close(fig)

if ALL_ADC_BY_NBITS.exists() or ALL_ADC_BY_NSUBQ.exists():
    print("=" * 60)
    print("All-experiments ADC timing (measure_all_adc.py)")
    print("=" * 60)
    if ALL_ADC_BY_NBITS.exists():
        _plot_all_adc_vs_param(ALL_ADC_BY_NBITS, "nbits", "nbits", DATASETS_TO_PLOT)
    if ALL_ADC_BY_NSUBQ.exists():
        _plot_all_adc_vs_param(ALL_ADC_BY_NSUBQ, "n_subquantizers", "M", DATASETS_TO_PLOT)

In [ ]:
# =============================================================================
# Paper-friendly ADC plots (no aggregation over heterogeneous configs)
# -----------------------------------------------------------------------------
# Reviewer note: Averaging ADC time over different M (or nbits) mixes very
# different compute regimes (e.g. M=1 vs M=96). The plots below avoid that:
# - Each point = one (M, nbits) config; error bars = run-to-run std (use --n_runs 20+ for tighter bars).
# - ADC vs bits_per_vector: natural x-axis (M×nbits); one curve per dataset.
# - ADC vs M, one line per nbits: shows structure without cross-nbits averaging.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

RESULTS_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/jz_relerr3")
FIGURES_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/figures")
ALL_ADC_PER_EXP = RESULTS_DIR / "all_adc_timing.csv"

# Log-x paper plots: max number of x tick labels showing real values (subsampling if needed). Use 3 for ~2–3 labels.
LOG_X_AXIS_MAX_LABELS = 3

# Per-dataset: which M and nbits to plot. None or missing key = all available.
M_TO_PLOT = {
    "deep": [1, 4, 8, 12, 24, 32, 96],
    "bigann": [1, 4, 8, 16, 32, 64, 128],
    "gist": [1, 8, 40, 60, 320, 480, 960],
    "msmarco": [1, 8, 32, 64, 256, 512, 1024],
    "openai": [1, 8, 32, 128, 256, 512, 1536],
}
NBITS_TO_PLOT = {}  # e.g. {"deep": [4, 6, 8]} or {} for all nbits

# paper_relerr_vs_adc_by_bpv: (M,nbits) on Pareto points only; optimal point is a red dot with no legend.

# Which file(s) to write: any subset of "pdf", "svg". Examples: ("pdf",) PDF only; ("svg",) SVG only; ("pdf", "svg") both.
FIGURE_SAVE_FORMATS = ("pdf", "svg")


def _save_figure(fig, path_pdf, formats=None):
    """Save using the stem of path_pdf (e.g. .../name.pdf -> name.pdf / name.svg)."""
    fmt_list = tuple(formats) if formats is not None else FIGURE_SAVE_FORMATS
    path_pdf = Path(path_pdf)
    parent = path_pdf.parent
    stem = path_pdf.stem
    saved = []
    for fmt in fmt_list:
        f = str(fmt).lower().lstrip(".")
        if f == "pdf":
            p = parent / f"{stem}.pdf"
            fig.savefig(p, dpi=300, bbox_inches="tight")
            saved.append(p)
        elif f == "svg":
            p = parent / f"{stem}.svg"
            fig.savefig(p, bbox_inches="tight")
            saved.append(p)
        else:
            raise ValueError(f"Unknown save format {fmt!r}; use 'pdf' and/or 'svg'")
    if saved:
        print(f"Saved {' and '.join(str(p) for p in saved)}")


def set_sci_axes(ax, tick_fontsize=40, offset_fontsize=30, scilimits=(-3, 3)):
    """
    Use scientific notation on linear axes when values are very small/large,
    and make the exponent/offset text (e.g. 1e-5) larger.

    Log-scaled axes are left with Matplotlib's log formatters. Applying
    ScalarFormatter + sci on a log axis produces misleading tick labels
    (e.g. 0 and 1 with a ×10³ offset instead of 1, 10, 100, …).
    """
    x_log = ax.get_xscale() == "log"
    y_log = ax.get_yscale() == "log"

    if not x_log:
        fmt_x = ScalarFormatter(useMathText=False)
        fmt_x.set_scientific(True)
        fmt_x.set_powerlimits(scilimits)
        ax.xaxis.set_major_formatter(fmt_x)

    if not y_log:
        fmt_y = ScalarFormatter(useMathText=False)
        fmt_y.set_scientific(False)
        fmt_y.set_powerlimits(scilimits)
        ax.yaxis.set_major_formatter(fmt_y)

    if not x_log and not y_log:
        ax.ticklabel_format(axis="both", style="sci", scilimits=scilimits, useMathText=False)
    elif not x_log:
        ax.ticklabel_format(axis="x", style="sci", scilimits=scilimits, useMathText=False)
    elif not y_log:
        ax.ticklabel_format(axis="y", style="sci", scilimits=scilimits, useMathText=False)

    ax.tick_params(labelsize=tick_fontsize)
    ax.xaxis.get_offset_text().set_fontsize(offset_fontsize)
    ax.yaxis.get_offset_text().set_fontsize(offset_fontsize)


def _log_axis_show_data_values(ax, values, axis="x", max_labels=LOG_X_AXIS_MAX_LABELS):
    """On a log scale, set tick positions/labels to actual data values (subset if many)."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v) & (v > 0)]
    if v.size == 0:
        return
    v = np.unique(np.sort(v))
    if v.size > max_labels:
        idx = np.unique(np.round(np.linspace(0, v.size - 1, max_labels)).astype(int))
        v = v[idx]
    lab = [str(int(x)) if abs(x - round(x)) < 1e-9 else str(x) for x in v]
    if axis == "x":
        ax.set_xticks(v)
        ax.set_xticklabels(lab)
    else:
        ax.set_yticks(v)
        ax.set_yticklabels(lab)


def _linear_axis_show_data_values(ax, values, axis="x"):
    """Linear axis: tick positions and labels at every unique data value (sorted)."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return
    v = np.unique(np.sort(v))
    lab = [str(int(x)) if abs(x - round(x)) < 1e-9 else str(x) for x in v]
    if axis == "x":
        ax.set_xticks(v)
        ax.set_xticklabels(lab)
    else:
        ax.set_yticks(v)
        ax.set_yticklabels(lab)


def plot_paper_adc_figures(datasets=None, m_to_plot=None, nbits_to_plot=None, save_formats=None):
    """Paper-ready ADC plots: per-config, no arbitrary averaging over M or nbits.
    Uses create_dynamic_color_map and create_dynamic_marker_map from notebook (run color cell first).
    save_formats: optional tuple overriding FIGURE_SAVE_FORMATS, e.g. ("pdf",) or ("svg",) or ("pdf", "svg").
    """
    try:
        create_dynamic_color_map([])
        create_dynamic_marker_map([])
    except NameError:
        raise NameError(
            "Run the 'Color and marker palettes' cell first "
            "(defines create_dynamic_color_map, create_dynamic_marker_map)"
        )

    if not ALL_ADC_PER_EXP.exists():
        print(f"⚠️  {ALL_ADC_PER_EXP} not found — run: python scripts/evals/measure_all_adc.py")
        return

    df = pd.read_csv(ALL_ADC_PER_EXP)
    m_cfg = m_to_plot if m_to_plot is not None else M_TO_PLOT
    nbits_cfg = nbits_to_plot if nbits_to_plot is not None else NBITS_TO_PLOT

    def _filter_sub(sub, dataset):
        out = sub.copy()
        m_sel = m_cfg.get(dataset) if isinstance(m_cfg, dict) else m_cfg
        if m_sel is not None:
            out = out[out["n_subquantizers"].astype(int).isin(m_sel)]
        nbits_sel = nbits_cfg.get(dataset) if isinstance(nbits_cfg, dict) else nbits_cfg
        if nbits_sel is not None:
            out = out[out["nbits"].astype(int).isin(nbits_sel)]
        return out

    if datasets is None:
        try:
            datasets = [d for d in DATASETS_TO_PLOT if d in df["dataset"].unique()]
        except NameError:
            datasets = sorted(df["dataset"].unique())
    else:
        datasets = [d for d in datasets if d in df["dataset"].unique()]

    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    formats = tuple(save_formats) if save_formats is not None else FIGURE_SAVE_FORMATS

    # --- (1) ADC vs bits_per_vector — one figure per dataset ---
    for dataset in datasets:
        sub = _filter_sub(df[df["dataset"] == dataset], dataset).sort_values("bits_per_vector")
        if sub.empty:
            continue

        fig, ax = plt.subplots()
        x = sub["bits_per_vector"].values
        y = sub["adc_cpu_time_pp_mean"].values
        yerr = sub["adc_cpu_time_pp_std"].fillna(0).values

        ax.errorbar(
            x, y, yerr=yerr,
            fmt="o",
            color=COLOR_PALETTE[0],
            markersize=12,
            linewidth=2,
            capsize=3,
            capthick=1.5,
            markeredgewidth=2,
            markeredgecolor="black"
        )
        ax.set_xlabel("Bits per vector", fontsize=40)
        ax.set_ylabel("ADC time (s)", fontsize=40)
        ax.set_xscale("log")
        set_sci_axes(ax, tick_fontsize=40, offset_fontsize=30)
        _log_axis_show_data_values(ax, sub["bits_per_vector"].values, axis="x", max_labels=LOG_X_AXIS_MAX_LABELS)

        ax.grid(alpha=0.8, axis="both", linestyle="--")
        for spine in ax.spines.values():
            spine.set_visible(False)

        plt.tight_layout()
        out = FIGURES_DIR / f"paper_adc_vs_bits_per_vector_{dataset}.pdf"
        _save_figure(fig, out, formats=formats)
        plt.show()
        plt.close(fig)

    # --- (2) LEGEND BY NBITS: ADC vs M, one line per nbits (per dataset) ---
    for dataset in datasets:
        sub = _filter_sub(df[df["dataset"] == dataset], dataset).sort_values(["n_subquantizers", "nbits"])
        if sub.empty:
            continue

        nbits_vals = sorted(sub["nbits"].unique())
        color_map = create_dynamic_color_map(nbits_vals)
        marker_map = create_dynamic_marker_map(nbits_vals)

        fig, ax = plt.subplots()
        for nb in nbits_vals:
            row = sub[sub["nbits"] == nb].sort_values("n_subquantizers")
            if row.empty:
                continue

            x = row["n_subquantizers"].values
            y = row["adc_cpu_time_pp_mean"].values
            yerr = row["adc_cpu_time_pp_std"].fillna(0).values

            ax.errorbar(
                x, y, yerr=yerr,
                fmt=marker_map[nb] + "-",
                label=f"nbits={int(nb)}",
                color=color_map[nb],
                markersize=12,
                linewidth=2,
                capsize=3,
                markeredgewidth=2,
                markeredgecolor="black"
            )

        ax.set_xlabel("nsubq", fontsize=40)
        ax.set_ylabel("ADC time (s)", fontsize=40)
        ax.set_xscale("log")
        set_sci_axes(ax, tick_fontsize=40, offset_fontsize=30)
        _log_axis_show_data_values(ax, sub["n_subquantizers"].values, axis="x", max_labels=3)

        ax.grid(alpha=0.8, axis="both", linestyle="--")
        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.legend(frameon=False, fontsize=24, loc="center left", bbox_to_anchor=(1.05, 0.5))
        plt.tight_layout()
        out = FIGURES_DIR / f"paper_adc_vs_M_by_nbits_{dataset}.pdf"
        _save_figure(fig, out, formats=formats)
        plt.show()
        plt.close(fig)

    # --- (3) LEGEND BY N_SUBQ (M): ADC vs nbits, one line per M (per dataset) ---
    for dataset in datasets:
        sub = _filter_sub(df[df["dataset"] == dataset], dataset).sort_values(["nbits", "n_subquantizers"])
        if sub.empty:
            continue

        m_vals = sorted(sub["n_subquantizers"].unique())
        color_map = create_dynamic_color_map(m_vals)
        marker_map = create_dynamic_marker_map(m_vals)

        fig, ax = plt.subplots()
        for m in m_vals:
            row = sub[sub["n_subquantizers"] == m].sort_values("nbits")
            if row.empty:
                continue

            x = row["nbits"].values
            y = row["adc_cpu_time_pp_mean"].values
            yerr = row["adc_cpu_time_pp_std"].fillna(0).values

            ax.errorbar(
                x, y, yerr=yerr,
                fmt=marker_map[m] + "-",
                label=f"M={int(m)}",
                color=color_map[m],
                markersize=12,
                linewidth=2,
                capsize=3,
                markeredgewidth=2,
                markeredgecolor="black"
            )

        ax.set_xlabel("nbits", fontsize=40)
        ax.set_ylabel("ADC time (s)", fontsize=40)
        set_sci_axes(ax, tick_fontsize=40, offset_fontsize=30)
        _linear_axis_show_data_values(ax, sub["nbits"].values, axis="x")

        ax.grid(alpha=0.8, axis="both", linestyle="--")
        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.legend(frameon=False, fontsize=24, loc="center left", bbox_to_anchor=(1.05, 0.5))
        plt.tight_layout()
        out = FIGURES_DIR / f"paper_adc_vs_nbits_by_M_{dataset}.pdf"
        _save_figure(fig, out, formats=formats)
        plt.show()
        plt.close(fig)

    # --- (4) Avg Relative Error vs ADC time (per dataset) ---
    if "rel_error_mean" not in df.columns:
        print("⚠️  rel_error_mean not in all_adc_timing.csv — skipping Avg Relative Error vs ADC time plot")
    else:
        def _plot_relerr_vs_adc(sub, group_col, label_fmt, out_suffix):
            group_vals = sorted(sub[group_col].unique())
            color_map = create_dynamic_color_map(group_vals)
            marker_map = create_dynamic_marker_map(group_vals)

            fig, ax = plt.subplots()
            for val in group_vals:
                row = sub[sub[group_col] == val]
                if row.empty:
                    continue

                x = row["adc_cpu_time_pp_mean"].values
                y = row["rel_error_mean"].values
                ax.scatter(
                    x, y,
                    label=label_fmt(val),
                    color=color_map[val],
                    marker=marker_map[val],
                    s=150,
                    edgecolors="black",
                    linewidths=2
                )

            ax.set_xlabel("ADC time (s)", fontsize=40)
            ax.set_ylabel("Avg Relative Error", fontsize=40)
            set_sci_axes(ax, tick_fontsize=40, offset_fontsize=30)

            ax.grid(alpha=0.8, axis="both", linestyle="--")
            for spine in ax.spines.values():
                spine.set_visible(False)

            ax.legend(frameon=False, fontsize=24, loc="center left", bbox_to_anchor=(1.05, 0.5))
            plt.tight_layout()
            return fig, ax

        for dataset in datasets:
            sub = _filter_sub(df[df["dataset"] == dataset], dataset)
            sub = sub.dropna(subset=["adc_cpu_time_pp_mean", "rel_error_mean"])
            if sub.empty:
                continue

            # 4a: Legend by nbits
            fig, ax = _plot_relerr_vs_adc(sub, "nbits", lambda v: f"nbits={int(v)}", "nbits")
            out = FIGURES_DIR / f"paper_relerr_vs_adc_by_nbits_{dataset}.pdf"
            _save_figure(fig, out, formats=formats)
            plt.show()
            plt.close(fig)

            # 4b: Legend by n_subq (M)
            fig, ax = _plot_relerr_vs_adc(sub, "n_subquantizers", lambda v: f"M={int(v)}", "M")
            out = FIGURES_DIR / f"paper_relerr_vs_adc_by_M_{dataset}.pdf"
            _save_figure(fig, out, formats=formats)
            plt.show()
            plt.close(fig)

            # 4c: (M,nbits) labels on Pareto (red) points only; optimal point is also a red dot, no legend.
            x_vals = sub["adc_cpu_time_pp_mean"].values
            y_vals = sub["rel_error_mean"].values
            m_arr = sub["n_subquantizers"].values.astype(int)
            b_arr = sub["nbits"].values.astype(int)
            n = len(x_vals)

            x_norm = (x_vals - x_vals.min()) / (x_vals.max() - x_vals.min() + 1e-12)
            y_norm = (y_vals - y_vals.min()) / (y_vals.max() - y_vals.min() + 1e-12)
            scores = x_norm + y_norm
            opt_idx = int(np.argmin(scores))

            order = np.argsort(x_vals)
            front = []
            best_y = np.inf
            for i in order:
                if y_vals[i] < best_y:
                    front.append(i)
                    best_y = y_vals[i]

            front_set = set(front)
            off_mask = np.array([i not in front_set for i in range(n)])

            fig, ax = plt.subplots(figsize=(10, 7))
            if off_mask.any():
                ax.scatter(
                    x_vals[off_mask], y_vals[off_mask],
                    color=COLOR_PALETTE[0],
                    marker="o",
                    s=120,
                    edgecolors="black",
                    linewidths=1.5,
                    zorder=1,
                )
            if front:
                xf = x_vals[front]
                yf = y_vals[front]
                ax.scatter(
                    xf, yf,
                    color="#D32F2F",
                    marker="o",
                    s=140,
                    edgecolors="black",
                    linewidths=1.5,
                    zorder=4,
                )

            ax.set_xlabel("ADC time (s)", fontsize=40, labelpad=16)
            ax.set_ylabel("Avg Relative Error", fontsize=40, labelpad=8)
            set_sci_axes(ax, tick_fontsize=40, offset_fontsize=30)
            ax.xaxis.get_offset_text().set_horizontalalignment("left")

            ax.grid(alpha=0.8, axis="both", linestyle="--")
            for spine in ax.spines.values():
                spine.set_visible(False)

            # Pareto labels: adjustText moves text into open (axes) space away from markers — no bbox patch.
            texts = []
            if front:
                xspan = float(x_vals.max() - x_vals.min()) + 1e-15
                yspan = float(y_vals.max() - y_vals.min()) + 1e-15
                fs = 17 if len(front) > 14 else 20
                for j, i in enumerate(front):
                    dx = 0.025 * xspan * (1 if j % 2 == 0 else -1)
                    dy = 0.012 * yspan * ((j // 2) % 3 - 1)
                    t = ax.text(
                        x_vals[i] + dx,
                        y_vals[i] + dy,
                        f"({m_arr[i]},{b_arr[i]})",
                        fontsize=fs,
                        color="black",
                        ha="center",
                        va="center",
                        zorder=5,
                    )
                    texts.append(t)
                try:
                    from adjustText import adjust_text

                    adjust_text(
                        texts,
                        x=[x_vals[i] for i in front],
                        y=[y_vals[i] for i in front],
                        ax=ax,
                        arrowprops=None,
                        expand_points=(2.5, 2.85),
                        expand_text=(1.28, 1.45),
                        force_points=(0.5, 0.75),
                        force_text=(0.65, 0.9),
                    )
                except ImportError:
                    pass

            ax.scatter(
                [x_vals[opt_idx]], [y_vals[opt_idx]], marker="o", s=220,
                color="#D32F2F", edgecolors="black", linewidths=1.5, zorder=10,
            )

            plt.tight_layout()
            out = FIGURES_DIR / f"paper_relerr_vs_adc_by_bpv_{dataset}.pdf"
            _save_figure(fig, out, formats=formats)
            plt.show()
            plt.close(fig)


# Examples: plot_paper_adc_figures(save_formats=("pdf",))  # PDF only
#           plot_paper_adc_figures(save_formats=("svg",))  # SVG only
plot_paper_adc_figures()

## Average distortion error (first 10k database vectors)

These cells plot the mean Euclidean reconstruction distortion, `||x - x_comp||`, for the same PQ models used by the CSVs above. The existing `*_PQ_reconstruction_error.csv` files already store this metric from `scripts/evals/run_evals.py` with `max_rec_samples=10_000`; if one is missing, the setup cell runs that evaluator first.


In [ ]:
# =============================================================================
# Average distortion error, ||x - x_comp||, over the first 10k database vectors
# =============================================================================
# `scripts/evals/run_evals.py` writes this as `reconstruction_error` and defines it as
# the mean Euclidean distance between original and reconstructed database vectors.

import subprocess
import sys

DISTORTION_DATASETS = ["deep", "bigann", "gist", "msmarco", "openai"]
DISTORTION_METHODS = ["PQ"]
DISTORTION_MAX_SAMPLES = 10_000
DISTORTION_FIGURES_DIR = DATA_DIR / "figures"
DISTORTION_Y_COL = "distortion_error"
DISTORTION_Y_LABEL = "Avg Distortion"
PROJECT_ROOT = Path("/home/cpanourg/projects/2-hdvc")
if not (PROJECT_ROOT / "scripts/evals/run_evals.py").exists():
    PROJECT_ROOT = Path.cwd()


def _distortion_csv_path(data_dir: Path, dataset: str, method: str) -> Path:
    return data_dir / f"{dataset}_{method}_reconstruction_error.csv"


def _distortion_input_csv_path(data_dir: Path, dataset: str, method: str) -> Path:
    return data_dir / f"{dataset}_{method}_adc_vs_exact_eval.csv"


def ensure_distortion_csvs(
    data_dir: Path,
    datasets: list,
    methods: list,
    max_samples: int = DISTORTION_MAX_SAMPLES,
):
    """Ensure reconstruction-error CSVs exist; compute them with run_evals.py if needed."""
    missing = []
    for dataset in datasets:
        for method in methods:
            out_csv = _distortion_csv_path(data_dir, dataset, method)
            needs_compute = True
            if out_csv.exists():
                try:
                    needs_compute = "reconstruction_error" not in pd.read_csv(out_csv, nrows=1).columns
                except pd.errors.EmptyDataError:
                    needs_compute = True
            if needs_compute:
                missing.append((dataset, method))

    if not missing:
        print("All reconstruction-error CSVs are present; using cached distortion values.")
        return

    for dataset, method in missing:
        input_csv = _distortion_input_csv_path(data_dir, dataset, method)
        if not input_csv.exists():
            raise FileNotFoundError(f"Missing input CSV for distortion calculation: {input_csv}")
        cmd = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/evals/run_evals.py"),
            "--input_csv",
            str(input_csv),
            "--output_dir",
            str(data_dir),
            "--eval_measures",
            "reconstruction_error",
            "--max_rec_samples",
            str(max_samples),
        ]
        print("Computing distortion:", " ".join(cmd))
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


def load_distortion_plot_df(
    data_dir: Path,
    datasets: list = DISTORTION_DATASETS,
    methods: list = DISTORTION_METHODS,
) -> pd.DataFrame:
    """Load cached/computed distortion values and average duplicate runs per config."""
    ensure_distortion_csvs(data_dir, datasets, methods)

    frames = []
    for dataset in datasets:
        for method in methods:
            path = _distortion_csv_path(data_dir, dataset, method)
            if not path.exists():
                continue
            df = pd.read_csv(path)
            if "dataset" not in df.columns:
                df["dataset"] = dataset
            if "method" not in df.columns:
                df["method"] = method
            frames.append(df)

    if not frames:
        raise FileNotFoundError(f"No reconstruction-error CSVs found in {data_dir}")

    dist_df = pd.concat(frames, ignore_index=True)
    dist_df = dist_df[
        dist_df["dataset"].isin(datasets) & dist_df["method"].isin(methods)
    ].copy()
    if "reconstruction_error" not in dist_df.columns:
        raise ValueError("Expected column 'reconstruction_error' in distortion CSVs")

    dist_df[DISTORTION_Y_COL] = dist_df["reconstruction_error"]

    # Some configs have repeated runs. Average them so each plotted point is one (M, nbits) config.
    group_cols = ["method", "dataset", "n_subquantizers", "nbits", "bits_per_vector"]
    agg_cols = {DISTORTION_Y_COL: "mean", "reconstruction_error": "mean"}
    for col in ["dim", "nb", "nb_sample", "train_size", "train_time_s", "encoding_time_s"]:
        if col in dist_df.columns:
            agg_cols[col] = "first"

    return (
        dist_df.groupby(group_cols, as_index=False)
        .agg(agg_cols)
        .sort_values(["dataset", "n_subquantizers", "nbits"])
    )


distortion_plot_df = load_distortion_plot_df(DATA_DIR)
distortion_plot_df.head()


In [ ]:
# Distortion vs nbits, one curve per n_subquantizers (same layout as relative-error plots)
plot_relerr_vs_x(
    distortion_plot_df,
    x_col="nbits",
    x_label="nbits",
    y_col=DISTORTION_Y_COL,
    y_label=DISTORTION_Y_LABEL,
    methods=DISTORTION_METHODS,
    datasets=DISTORTION_DATASETS,
    group_by="n_subquantizers",
    output_dir=DISTORTION_FIGURES_DIR,
    nbits_subquantizers=NBITS_PLOT_SUBQUANTIZERS,
)


In [ ]:
# Distortion vs nsubq, one curve per nbits (same layout as relative-error plots)
plot_relerr_vs_x(
    distortion_plot_df,
    x_col="n_subquantizers",
    x_label="nsubq",
    y_col=DISTORTION_Y_COL,
    y_label=DISTORTION_Y_LABEL,
    methods=DISTORTION_METHODS,
    datasets=DISTORTION_DATASETS,
    group_by="nbits",
    output_dir=DISTORTION_FIGURES_DIR,
    num_subq_plot_subquantizers=NUM_SUBQ_PLOT_SUBQUANTIZERS,
)
